# 04 · Вызов инструментов

Модель ничего не вызывает — она порождает строку определённого вида. Разбирает строку, выполняет функцию и возвращает результат ваш код. Механика — `books/00-basics.pdf`, раздел 4.

Учим четырём вещам сразу, и три из них — про то, когда **не** вызывать:

| группа | поведение |
|---|---|
| `tool` | вызвать |
| `direct` | ответ уже есть — не вызывать |
| `stop` | результат получен — ответить, не звать снова |
| `multi`, `empty` | цепочка из двух; пустой результат |

In [ ]:
from common import MODEL_ID, DATA, RUNS, tools_suite, fmt, read_raw

import torch
from transformers import AutoModelForImageTextToText, AutoProcessor, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model
from vlmkit import ChatCollator, load_jsonl, memory_report, preview, evaluate as ev
from vlmkit.compat import supported, first_accepted
from vlmkit.toolcalls import detect_style, parse_tool_calls, strip_thinking

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, dtype=torch.bfloat16, device_map={"": 0},
    attn_implementation="sdpa", trust_remote_code=True,
)
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True, max_pixels=1003520)

print("формат вызова у модели:", detect_style(processor.tokenizer.chat_template))
print(memory_report())

## До

Смотрим не только на метрику, но и на сырой вывод: что модель реально пишет и разбирается ли это парсером.

In [ ]:
suite = tools_suite()
tools = load_jsonl(DATA / "tools.jsonl")
tools_raw = read_raw("tools.jsonl")

before_metrics = ev.run(model, processor, suite)
print("до:", fmt(before_metrics))

# Сырой вывод на одном примере из группы tool и одном из direct
for want in ("tool", "direct"):
    sample = next(s for s, r in zip(suite.samples, suite.groups) if r == want)
    out = ev.generate(model, processor, [sample], max_new_tokens=200)[0]
    print(f"\n[{want}] {sample.messages[1]['content'][0]['text'][:60]}")
    print("  сырой:", repr(out[:150]))
    print("  разобрано:", parse_tool_calls(out))

## Что попадает в градиент

На траектории `multi` три вещи должны быть видны сразу: обе реплики ассистента открыты, результаты инструментов закрыты, блок `<think>` закрыт.

In [ ]:
multi = next(s for s, r in zip(tools, tools_raw) if r["group"] == "multi")
print(preview(multi, processor))

## Обучение

Одна эпоха, не три. На 33 траекториях три эпохи в прошлый раз дали ложные срабатывания 47%: модель научилась вызывать и стала вызывать везде.

In [ ]:
lora = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=r"^(?!.*(visual|vision)).*(q_proj|k_proj|v_proj|o_proj|gate_proj|up_proj|down_proj)$",
    use_rslora=True, bias="none", task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora)
model.enable_input_require_grads()
model.config.use_cache = False

args = dict(
    output_dir=str(RUNS / "sft-tools"),
    per_device_train_batch_size=1, gradient_accumulation_steps=8,
    num_train_epochs=1, learning_rate=1e-4, lr_scheduler_type="cosine",
    bf16=True, gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="adamw_torch_fused", logging_steps=5, save_strategy="no",
    remove_unused_columns=False, report_to=[], seed=42,
)
args |= first_accepted(TrainingArguments, {"warmup_ratio": 0.05, "warmup_steps": 2})

trainer = Trainer(
    model=model, args=TrainingArguments(**supported(TrainingArguments, args)),
    train_dataset=tools[::2], data_collator=ChatCollator(processor),
)
trainer.train()

## После

In [ ]:
model.eval()
after_metrics = ev.run(model, processor, suite)
print(f"до:    {fmt(before_metrics)}")
print(f"после: {fmt(after_metrics)}")

for want in ("tool", "direct", "stop"):
    sample = next(s for s, r in zip(suite.samples, suite.groups) if r == want)
    out = ev.generate(model, processor, [sample], max_new_tokens=200)[0]
    print(f"\n[{want}] → {parse_tool_calls(out) or strip_thinking(out)[:120]}")

model.save_pretrained(str(RUNS / "sft-tools"))

## Полный цикл

Тот самый `while`: модель пишет строку → код разбирает → выполняет → приписывает результат → вызывает модель снова. Рассуждение выбрасывается из истории между витками. Развёрнутая версия — `tools/agent_loop.py`.

In [ ]:
FAKE_TOOLS = {
    "read_document": lambda section_id: f"Раздел {section_id}: обзор утверждает, что формат обучения на успеваемость не влияет.",
    "select_skill":  lambda name: f"Методика {name}: проверить связь с проблемой, проверяемость, опровержимость.",
    "search_sources": lambda query: "source_733: метаанализ 2023, 61 исследование.",
    "read_source":   lambda source_id: f"{source_id}: средний эффект d = 0.07, незначим.",
    "search_chat_history": lambda query: "совпадений не найдено.",
}
SYSTEM_TOOLS = tools[0].messages[0]["content"][0]["text"]   # тот же список инструментов, что в данных

messages = [
    {"role": "system", "content": [{"type": "text", "text": SYSTEM_TOOLS}]},
    {"role": "user",   "content": [{"type": "text", "text": "Не противоречит ли моя третья глава обзору?"}]},
]

for step in range(5):
    prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[prompt], return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=300, do_sample=False)
    reply = processor.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True)

    calls = parse_tool_calls(reply)
    clean = strip_thinking(reply)
    messages.append({"role": "assistant", "content": [{"type": "text", "text": clean}]})
    print(f"шаг {step+1}: {'вызов ' + str([c['name'] for c in calls]) if calls else 'ответ: ' + clean[:100]}")

    if not calls:
        break
    for c in calls:
        result = FAKE_TOOLS.get(c["name"], lambda **k: "нет такого инструмента")(**c["arguments"])
        messages.append({"role": "tool", "content": [{"type": "text", "text": result}]})
else:
    print("вышли по лимиту — модель не остановилась сама")

## На что смотреть

**Попадание выросло, ложные тоже** — модель дёргает инструменты везде. Больше примеров `direct`, меньше эпох.

**Цикл вышел по лимиту** — модель не умеет останавливаться. Это отдельное поведение, ему учит группа `stop`.

**Парсер вернул пусто, а в сыром выводе вызов есть** — формат не совпал. `detect_style` показывает, чего ждёт модель; `STYLE` в `data/build_tools.py` должен совпадать.